In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta
import uuid
import time
import json
import requests
from IPython.display import clear_output
from pyspark.sql import functions as F
from pyspark.sql.types import *
from notebookutils import mssparkutils

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 3, Finished, Available, Finished)

In [2]:
# Load policy data with date ranges
policy_df = pd.read_csv("/lakehouse/default/Files/AutoClaims_csv/policy.csv")

# Convert date columns to datetime
policy_df['start_date'] = pd.to_datetime(policy_df['start_date'])
policy_df['end_date'] = pd.to_datetime(policy_df['end_date'])

# Rename for consistency
policy_df = policy_df.rename(columns={'vehicle_vin': 'vin'})

# Sort by start_date to process chronologically
policy_df = policy_df.sort_values(['policyholder_id', 'vin', 'start_date'])

print(f"Loaded {len(policy_df)} policies")
print(f"Date range: {policy_df['start_date'].min()} to {policy_df['end_date'].max()}")
print(f"\nUnique policyholders: {policy_df['policyholder_id'].nunique()}")
print(f"Unique vehicles: {policy_df['vin'].nunique()}")
display(policy_df.head(20))

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 4, Finished, Available, Finished)

Loaded 82 policies
Date range: 2015-02-20 00:00:00 to 2026-01-01 00:00:00

Unique policyholders: 14
Unique vehicles: 28


SynapseWidget(Synapse.DataFrame, 06409554-5fa5-4991-9252-b898a404c694)

## Configure Fabric Eventhouse Connection (Native)
Configure the connection details for your Fabric Eventhouse using native Fabric capabilities.

In [3]:
# Configure Fabric Eventhouse connection (Native Fabric approach)
# Replace these values with your actual Eventhouse and Eventstream details

EVENTHOUSE_WORKSPACE = "WS_AutoClaimsPOC"  # Your Fabric workspace name
EVENTHOUSE_NAME = "ev_driver_telemetry"  # Your Eventhouse name
EVENTHOUSE_DATABASE = "ev_driver_telemetry"  # Your database name in Eventhouse
EVENTHOUSE_TABLE = "driver_telemetry_data"  # Target table name

# For streaming to Eventstream (optional - if you want to use Eventstream before Eventhouse)
EVENTSTREAM_WORKSPACE = "YourWorkspaceName"  # Your workspace name
EVENTSTREAM_NAME = "YourEventstream"  # Your Eventstream name

# Build connection strings for Fabric
# eventhouse_connection = f"https://trd-qstp674t3q2kk676wc.z3.kusto.fabric.microsoft.com"
# Build connection strings for Fabric - using the ingest-<clustername> format for ingestion
eventhouse_query_uri = f"https://trd-qstp674t3q2kk676wc.z3.kusto.fabric.microsoft.com"
eventhouse_ingest_uri = f"https://ingest-trd-qstp674t3q2kk676wc.z3.kusto.fabric.microsoft.com"



print(f"✓ Fabric Eventhouse connection configured")
print(f"  Workspace: {EVENTHOUSE_WORKSPACE}")
print(f"  Eventhouse: {EVENTHOUSE_NAME}")
print(f"  Database: {EVENTHOUSE_DATABASE}")
print(f"  Table: {EVENTHOUSE_TABLE}")
#print(f"  Connection: {eventhouse_connection}")
print(f"  Query URI: {eventhouse_query_uri}")
print(f"  Ingest URI: {eventhouse_ingest_uri}")
print(f"\n⚠ Note: Ensure your workspace has proper permissions to access the Eventhouse")
print(f"  You may need to grant 'Admin' or 'Contributor' role on the Eventhouse")

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 5, Finished, Available, Finished)

✓ Fabric Eventhouse connection configured
  Workspace: WS_AutoClaimsPOC
  Eventhouse: ev_driver_telemetry
  Database: ev_driver_telemetry
  Table: driver_telemetry_data
  Query URI: https://trd-qstp674t3q2kk676wc.z3.kusto.fabric.microsoft.com
  Ingest URI: https://ingest-trd-qstp674t3q2kk676wc.z3.kusto.fabric.microsoft.com

⚠ Note: Ensure your workspace has proper permissions to access the Eventhouse
  You may need to grant 'Admin' or 'Contributor' role on the Eventhouse


## Create Eventhouse Table Schema
Creates the driver_telemetry_data table in Eventhouse if it doesn't exist.

In [4]:
# Create Eventhouse table if it doesn't exist
# Using KQL commands to create the table schema

kql_create_table_if_not_exists = f"""
.create-merge table {EVENTHOUSE_TABLE} (
    Trip_ID: string,
    policyholder_id: int,
    vin: string,
    Start_Time: datetime,
    End_Time: datetime,
    Start_Location_Lat: real,
    Start_Location_Lon: real,
    Time_of_Day_Category: string,
    Duration_Min: real,
    Distance_Miles: real,
    Peak_High_Speed: real,
    Average_Speed: real,
    Peak_Low_Speed: real,
    Max_Speed_Limit_Exceeded: real,
    Sudden_Braking_Count: int,
    Rapid_Acceleration_Count: int,
    Harsh_Cornering_Count: int,
    Speeding_Incidents: int,
    Safety_Score: real,
    Braking_Index: real,
    Acceleration_Index: real,
    Trip_Risk_Level: string
)
"""

print("=" * 80)
print("KQL Command to Create/Update Table in Eventhouse")
print("=" * 80)
print("Please run this command in your Eventhouse Query Editor to create the table:")
print()
print(kql_create_table_if_not_exists)
print("=" * 80)

try:
    # Try to verify table exists by attempting a simple query with proper authentication
    print("\nAttempting to verify table existence...")
    
    test_df = spark.read \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", eventhouse_query_uri) \
        .option("kustoDatabase", EVENTHOUSE_DATABASE) \
        .option("kustoQuery", f"{EVENTHOUSE_TABLE} | count") \
        .option("accessToken", mssparkutils.credentials.getToken(eventhouse_query_uri)) \
        .load()
    
    row_count = test_df.collect()[0][0]
    print(f"✓ Table exists and is accessible")
    print(f"  Current row count: {row_count}")
    
except Exception as e:
    print(f"⚠ Could not verify table existence: {str(e)}")
    print(f"\nTroubleshooting steps:")
    print(f"1. Create the table manually using the KQL command shown above")
    print(f"2. Ensure your workspace has 'Admin' or 'Contributor' permissions on the Eventhouse")
    print(f"3. Verify the Eventhouse name and database name are correct")
    print(f"4. Check that the Eventhouse is in the same workspace or you have cross-workspace access")
    print(f"\nYou can continue - the code will attempt to write data anyway.")

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 6, Finished, Available, Finished)

KQL Command to Create/Update Table in Eventhouse
Please run this command in your Eventhouse Query Editor to create the table:


.create-merge table driver_telemetry_data (
    Trip_ID: string,
    policyholder_id: int,
    vin: string,
    Start_Time: datetime,
    End_Time: datetime,
    Start_Location_Lat: real,
    Start_Location_Lon: real,
    Time_of_Day_Category: string,
    Duration_Min: real,
    Distance_Miles: real,
    Peak_High_Speed: real,
    Average_Speed: real,
    Peak_Low_Speed: real,
    Max_Speed_Limit_Exceeded: real,
    Sudden_Braking_Count: int,
    Rapid_Acceleration_Count: int,
    Harsh_Cornering_Count: int,
    Speeding_Incidents: int,
    Safety_Score: real,
    Braking_Index: real,
    Acceleration_Index: real,
    Trip_Risk_Level: string
)


Attempting to verify table existence...
✓ Table exists and is accessible
  Current row count: 0


In [5]:
# Define driver behavior profiles
DRIVER_PROFILES = {
    'safe': {
        'avg_speed_range': (30, 50),
        'peak_speed_range': (45, 65),
        'speed_limit_exceed_range': (0, 5),
        'speeding_incidents_range': (0, 2),
        'sudden_braking_range': (0, 3),
        'rapid_accel_range': (0, 2),
        'harsh_cornering_range': (0, 2),
        'safety_score_range': (80, 100),
        'braking_index_range': (0.1, 0.3),
        'acceleration_index_range': (0.1, 0.3),
        'risk_level': 'Low'
    },
    'aggressive': {
        'avg_speed_range': (55, 75),
        'peak_speed_range': (80, 110),
        'speed_limit_exceed_range': (15, 40),
        'speeding_incidents_range': (5, 15),
        'sudden_braking_range': (8, 20),
        'rapid_accel_range': (6, 18),
        'harsh_cornering_range': (5, 15),
        'safety_score_range': (20, 50),
        'braking_index_range': (0.7, 0.95),
        'acceleration_index_range': (0.7, 0.95),
        'risk_level': 'High'
    },
    'moderate': {
        'avg_speed_range': (45, 60),
        'peak_speed_range': (65, 80),
        'speed_limit_exceed_range': (5, 15),
        'speeding_incidents_range': (2, 6),
        'sudden_braking_range': (3, 8),
        'rapid_accel_range': (2, 7),
        'harsh_cornering_range': (2, 6),
        'safety_score_range': (55, 75),
        'braking_index_range': (0.35, 0.65),
        'acceleration_index_range': (0.35, 0.65),
        'risk_level': 'Medium'
    }
}

TIME_OF_DAY = ['Morning', 'Afternoon', 'Evening', 'Night']

def assign_driver_profile(policyholder_id):
    """Assign a consistent driver profile based on policyholder_id"""
    # Use modulo to create consistent distribution
    profile_id = policyholder_id % 3
    if profile_id == 0:
        return 'safe'
    elif profile_id == 1:
        return 'aggressive'
    else:
        return 'moderate'

def generate_trip(policyholder_id, vin, trip_number, base_date, profile_type):
    """Generate a single trip record"""
    profile = DRIVER_PROFILES[profile_type]
    
    # Generate trip ID
    trip_id = f"{policyholder_id}_{vin[-6:]}_{trip_number:04d}"
    
    # Generate start time (spread over 6 months)
    days_offset = random.randint(0, 180)
    hours_offset = random.randint(0, 23)
    minutes_offset = random.randint(0, 59)
    start_time = base_date + timedelta(days=days_offset, hours=hours_offset, minutes=minutes_offset)
    
    # Generate duration (5 to 120 minutes)
    duration_min = round(random.uniform(5, 120), 2)
    end_time = start_time + timedelta(minutes=duration_min)
    
    # Generate location (US coordinates roughly)
    start_lat = round(random.uniform(25.0, 48.0), 6)
    start_lon = round(random.uniform(-125.0, -65.0), 6)
    
    # Time of day based on hour
    hour = start_time.hour
    if 5 <= hour < 12:
        time_of_day = 'Morning'
    elif 12 <= hour < 17:
        time_of_day = 'Afternoon'
    elif 17 <= hour < 21:
        time_of_day = 'Evening'
    else:
        time_of_day = 'Night'
    
    # Generate distance based on duration and speed
    avg_speed = round(random.uniform(*profile['avg_speed_range']), 2)
    distance_miles = round((duration_min / 60) * avg_speed, 2)
    
    # Speed metrics
    peak_high_speed = round(random.uniform(*profile['peak_speed_range']), 2)
    peak_low_speed = round(random.uniform(0, 20), 2)
    max_speed_limit_exceeded = round(random.uniform(*profile['speed_limit_exceed_range']), 2)
    
    # Event counts
    speeding_incidents = random.randint(*profile['speeding_incidents_range'])
    sudden_braking_count = random.randint(*profile['sudden_braking_range'])
    rapid_accel_count = random.randint(*profile['rapid_accel_range'])
    harsh_cornering_count = random.randint(*profile['harsh_cornering_range'])
    
    # Calculated metrics
    safety_score = round(random.uniform(*profile['safety_score_range']), 2)
    braking_index = round(random.uniform(*profile['braking_index_range']), 2)
    acceleration_index = round(random.uniform(*profile['acceleration_index_range']), 2)
    risk_level = profile['risk_level']
    
    return {
        'Trip_ID': trip_id,
        'policyholder_id': policyholder_id,
        'vin': vin,
        'Start_Time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
        'End_Time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        'Start_Location_Lat': start_lat,
        'Start_Location_Lon': start_lon,
        'Time_of_Day_Category': time_of_day,
        'Duration_Min': duration_min,
        'Distance_Miles': distance_miles,
        'Peak_High_Speed': peak_high_speed,
        'Average_Speed': avg_speed,
        'Peak_Low_Speed': peak_low_speed,
        'Max_Speed_Limit_Exceeded': max_speed_limit_exceeded,
        'Sudden_Braking_Count': sudden_braking_count,
        'Rapid_Acceleration_Count': rapid_accel_count,
        'Harsh_Cornering_Count': harsh_cornering_count,
        'Speeding_Incidents': speeding_incidents,
        'Safety_Score': safety_score,
        'Braking_Index': braking_index,
        'Acceleration_Index': acceleration_index,
        'Trip_Risk_Level': risk_level
    }



StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 7, Finished, Available, Finished)

## Policy-Aligned Telemetry with Fabric Eventhouse REST API Integration
This cell generates driver telemetry data aligned with actual policy coverage periods and **automatically sends** each batch to your Fabric Eventhouse using **Kusto REST API**.

**Features:**
- **REST API Integration** - Uses Kusto streaming ingest REST API (`/v1/rest/ingest/{database}/{table}`)
- **Native Fabric Authentication** - Uses `mssparkutils.credentials.getToken()` for authentication
- **MultiJSON Format** - Sends data as newline-delimited JSON for efficient batch ingestion
- Uses dates from policy start_date and end_date (not system time)
- **Processes ALL policies every time** (no policy filtering)
- **Writes directly to Eventhouse** using HTTP POST requests
- **Also stores in Delta table** for local tracking (optional)
- **Prevents duplicate/overlapping trip times** for the same policyholder/vehicle
- Checks existing trip Start_Time and End_Time from the table
- Skips generating a new trip if it would overlap with any existing trip time
- Generates trips that fall within active policy coverage periods
- Processes policies in batches every 5 seconds
- Displays live updates showing progress and ingestion status
- Press **Interrupt Kernel** (⏹️) to stop generation

**REST API Details:**
- Endpoint: `POST https://ingest-{eventhouse}.z1.kusto.fabric.microsoft.com/v1/rest/ingest/{database}/{table}?streamFormat=MultiJSON`
- Authentication: Bearer token from Fabric workspace identity
- Content-Type: `application/json`
- Data format: Newline-delimited JSON (MultiJSON)

**Safe to run multiple times** - will generate new trips for all policies while preventing any time-based overlaps with existing trips.


In [ ]:
# Generate telemetry data and send to Fabric Eventhouse using REST API
# Ensures no duplicate or overlapping trip times for same policyholder/vin
telemetry_records = []
trip_counter = {}  # Track trip numbers per vehicle
policy_index = 0  # Track which policy we're processing

print("Starting telemetry generation aligned with policy periods...")
print(f"Total policies: {len(policy_df)}")

# Load existing trip timestamps to prevent time-based duplicates
existing_trips = {}
try:
    existing_data = spark.sql("""
        SELECT policyholder_id, vin, Start_Time, End_Time
        FROM driver_telemetry_data
    """).toPandas()
    
    if len(existing_data) > 0:
        print(f"\nFound {len(existing_data)} existing trips in the table")
        
        # Group by policyholder_id and vin for fast lookup
        for _, row in existing_data.iterrows():
            key = f"{row['policyholder_id']}_{row['vin']}"
            if key not in existing_trips:
                existing_trips[key] = []
            existing_trips[key].append({
                'start': pd.to_datetime(row['Start_Time']),
                'end': pd.to_datetime(row['End_Time'])
            })
        print(f"Loaded existing trips for {len(existing_trips)} vehicle/policyholder combinations")
    else:
        print("\nNo existing trips found - will generate fresh data")
    
except Exception as e:
    print(f"\nNo existing data found or error: {e}")
    print("Processing all policies as new...")

policies_to_process = policy_df.reset_index(drop=True)

print(f"\nTotal policies to process: {len(policies_to_process)}")
print(f"Generating trips every 5 seconds and sending to Fabric Eventhouse via REST API...")
print(f"Press 'Interrupt Kernel' to stop\n")

# Get authentication token once at the start
try:
    access_token = mssparkutils.credentials.getToken(eventhouse_query_uri)
    print("✓ Authentication token obtained successfully")
except Exception as e:
    print(f"⚠ Failed to get authentication token: {e}")
    access_token = None

try:
    iteration = 0
    total_skipped = 0
    total_sent_to_eventhouse = 0
    
    while policy_index < len(policies_to_process):
        iteration += 1
        
        # Select a batch of active policies (simulate 5-15 concurrent policies)
        batch_size = min(random.randint(5, 15), len(policies_to_process) - policy_index)
        batch_policies = policies_to_process.iloc[policy_index:policy_index + batch_size]
        
        new_trips = []
        skipped_in_batch = 0
        
        for idx, policy_row in batch_policies.iterrows():
            policyholder_id = policy_row['policyholder_id']
            vin = policy_row['vin']
            policy_start = policy_row['start_date']
            policy_end = policy_row['end_date']
            policy_number = policy_row['policy_number']
            
            # Get existing trips for this vehicle/policyholder
            vehicle_key = f"{policyholder_id}_{vin}"
            existing_vehicle_trips = existing_trips.get(vehicle_key, [])
            
            # Generate 2-5 trips per policy period per iteration
            num_trips = random.randint(2, 5)
            
            for trip_num in range(num_trips):
                # Track trip number for each vehicle
                if vehicle_key not in trip_counter:
                    trip_counter[vehicle_key] = 0
                trip_counter[vehicle_key] += 1
                
                # Assign driver profile based on policyholder_id
                profile_type = assign_driver_profile(policyholder_id)
                profile = DRIVER_PROFILES[profile_type]
                
                # Generate random date/time within policy period
                policy_duration_days = (policy_end - policy_start).days
                if policy_duration_days > 0:
                    random_day_offset = random.randint(0, policy_duration_days - 1)
                    random_hour = random.randint(0, 23)
                    random_minute = random.randint(0, 59)
                    start_time = policy_start + timedelta(days=random_day_offset, hours=random_hour, minutes=random_minute)
                else:
                    start_time = policy_start
                
                # Generate trip duration (5 to 60 minutes)
                duration_min = round(random.uniform(5, 60), 2)
                end_time = start_time + timedelta(minutes=duration_min)
                
                # Ensure trip doesn't exceed policy end date
                if end_time > policy_end:
                    end_time = policy_end
                    duration_min = (end_time - start_time).total_seconds() / 60
                
                # Check if this trip time overlaps with any existing trip for this vehicle
                has_overlap = False
                for existing_trip in existing_vehicle_trips:
                    # Check for any overlap: new trip starts before existing ends AND new trip ends after existing starts
                    if start_time < existing_trip['end'] and end_time > existing_trip['start']:
                        has_overlap = True
                        break
                
                # Skip this trip if it overlaps with an existing one
                if has_overlap:
                    skipped_in_batch += 1
                    total_skipped += 1
                    continue
                
                # Generate deterministic Trip_ID including policy number and timestamp
                trip_timestamp = int(start_time.timestamp())
                trip_id = f"{policyholder_id}_{vin[-6:]}_{policy_number}_{trip_timestamp}_{trip_counter[vehicle_key]:04d}"
                
                # Generate location (US coordinates roughly)
                start_lat = round(random.uniform(25.0, 48.0), 6)
                start_lon = round(random.uniform(-125.0, -65.0), 6)
                
                # Time of day based on hour
                hour = start_time.hour
                if 5 <= hour < 12:
                    time_of_day = 'Morning'
                elif 12 <= hour < 17:
                    time_of_day = 'Afternoon'
                elif 17 <= hour < 21:
                    time_of_day = 'Evening'
                else:
                    time_of_day = 'Night'
                
                # Generate distance based on duration and speed
                avg_speed = round(random.uniform(*profile['avg_speed_range']), 2)
                distance_miles = round((duration_min / 60) * avg_speed, 2)
                
                # Speed metrics
                peak_high_speed = round(random.uniform(*profile['peak_speed_range']), 2)
                peak_low_speed = round(random.uniform(0, 20), 2)
                max_speed_limit_exceeded = round(random.uniform(*profile['speed_limit_exceed_range']), 2)
                
                # Event counts
                speeding_incidents = random.randint(*profile['speeding_incidents_range'])
                sudden_braking_count = random.randint(*profile['sudden_braking_range'])
                rapid_accel_count = random.randint(*profile['rapid_accel_range'])
                harsh_cornering_count = random.randint(*profile['harsh_cornering_range'])
                
                # Calculated metrics
                safety_score = round(random.uniform(*profile['safety_score_range']), 2)
                braking_index = round(random.uniform(*profile['braking_index_range']), 2)
                acceleration_index = round(random.uniform(*profile['acceleration_index_range']), 2)
                risk_level = profile['risk_level']
                
                trip_data = {
                    'Trip_ID': trip_id,
                    'policyholder_id': policyholder_id,
                    'vin': vin,
                    'Start_Time': start_time.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
                    'End_Time': end_time.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
                    'Start_Location_Lat': start_lat,
                    'Start_Location_Lon': start_lon,
                    'Time_of_Day_Category': time_of_day,
                    'Duration_Min': duration_min,
                    'Distance_Miles': distance_miles,
                    'Peak_High_Speed': peak_high_speed,
                    'Average_Speed': avg_speed,
                    'Peak_Low_Speed': peak_low_speed,
                    'Max_Speed_Limit_Exceeded': max_speed_limit_exceeded,
                    'Sudden_Braking_Count': sudden_braking_count,
                    'Rapid_Acceleration_Count': rapid_accel_count,
                    'Harsh_Cornering_Count': harsh_cornering_count,
                    'Speeding_Incidents': speeding_incidents,
                    'Safety_Score': safety_score,
                    'Braking_Index': braking_index,
                    'Acceleration_Index': acceleration_index,
                    'Trip_Risk_Level': risk_level
                }
                
                new_trips.append(trip_data)
                telemetry_records.append(trip_data)
                
                # Add to existing trips cache to prevent duplicates within this session
                existing_vehicle_trips.append({
                    'start': start_time,
                    'end': end_time
                })
        
        #print(f"\nlength of new_trips: {len(new_trips)}")
        # Send new trips to Fabric Eventhouse using REST API
        if len(new_trips) > 0:
            # Build the REST API endpoint URL
            ingest_url = f"{eventhouse_query_uri}/v1/rest/ingest/{EVENTHOUSE_DATABASE}/{EVENTHOUSE_TABLE}"
            print(f"\ningest_url: {ingest_url}")
            # Add query parameters for JSON format
            params = {
                'streamFormat': 'MultiJSON'
            }
            
            # Extract just the hostname from the URI
            hostname = eventhouse_query_uri.replace('https://', '').replace('http://', '').rstrip('/')

            # Prepare headers with authentication
            headers = {
                'Authorization': f'Bearer {access_token}',
                'Content-Type': 'application/json',
                'Accept': 'application/json',
                #'Host': eventhouse_ingest_uri.replace('https://', '').replace('http://', ''),
                #'Host': ingest_url.replace('https://', '').replace('http://', ''),
                'Host': hostname,
                'x-ms-client-request-id': f'Notebook-{iteration}-{int(time.time())}',
                'x-ms-app': 'FabricNotebook',
                'Connection': 'Keep-Alive'
            }
            
            # Convert trips to newline-delimited JSON (MultiJSON format)
            json_data = '\n'.join([json.dumps(trip) for trip in new_trips])

            # Debug: Print request details for first iteration
            if iteration == 1:
                print("\n=== DEBUG: Request Details ===")
                print(f"URL: {ingest_url}")
                print(f"Params: {params}")
                print(f"Hostname: {hostname}")
                print(f"Data sample (first 500 chars): {json_data[:500]}")
                print(f"Total trips in batch: {len(new_trips)}")
                print("=" * 60 + "\n")

            
            # Send POST request to Eventhouse
            response = requests.post(
                ingest_url,
                params=params,
                headers=headers,
                data=json_data.encode('utf-8'),
                timeout=30
            )
            
            # Check response - stop on error
            if response.status_code != 200:
                #error_msg = response.text
                print(f"\n❌ ERROR: Failed to send data to Eventhouse (HTTP {response.status_code})")
                #print(f"Error message: {error_msg}")
                
                if response.status_code == 403:
                    print("\n📋 PERMISSION ERROR")
                    print("=" * 80)
                    print("The workspace doesn't have permission to write to the Eventhouse.")
                    print("\nTo fix this:")
                    print("1. Go to your Eventhouse in Fabric")
                    print("2. Click 'Manage permissions'")
                    print("3. Add your workspace or user with 'Admin' or 'Contributor' role")
                    print("4. Wait a few minutes for permissions to propagate")
                    print("5. Re-run this cell")
                    print("=" * 80)
                elif response.status_code == 404:
                    print("\n📋 TABLE OR DATABASE NOT FOUND")
                    print("Please verify the database and table names, and create the table using the KQL command from cell 6")
                
                print("\n⛔ STOPPING EXECUTION - Fix the error before continuing")
                #raise Exception(f"Eventhouse ingestion failed with HTTP {response.status_code}: {error_msg}")
                raise Exception(f"Eventhouse ingestion failed with HTTP {response.status_code}: {response.text}")
            
            total_sent_to_eventhouse += len(new_trips)

        
        # Also insert into Delta table for local tracking
        if len(new_trips) > 0:
            # Convert string timestamps back to datetime for Delta table
            trips_for_delta = []
            for trip in new_trips:
                trip_copy = trip.copy()
                trip_copy['Start_Time'] = pd.to_datetime(trip['Start_Time'])
                trip_copy['End_Time'] = pd.to_datetime(trip['End_Time'])
                trips_for_delta.append(trip_copy)
            
            batch_df = pd.DataFrame(trips_for_delta)
            
            # Convert to Spark DataFrame with proper types
            spark_batch_df = spark.createDataFrame(batch_df)
            spark_batch_df = spark_batch_df.select(
                F.col("Trip_ID").cast("string"),
                F.col("policyholder_id").cast("int"),
                F.col("vin").cast("string"),
                F.col("Start_Time").cast("timestamp"),
                F.col("End_Time").cast("timestamp"),
                F.col("Start_Location_Lat").cast("decimal(9,6)"),
                F.col("Start_Location_Lon").cast("decimal(9,6)"),
                F.col("Time_of_Day_Category").cast("string"),
                F.col("Duration_Min").cast("decimal(10,2)"),
                F.col("Distance_Miles").cast("decimal(10,2)"),
                F.col("Peak_High_Speed").cast("decimal(5,2)"),
                F.col("Average_Speed").cast("decimal(5,2)"),
                F.col("Peak_Low_Speed").cast("decimal(5,2)"),
                F.col("Max_Speed_Limit_Exceeded").cast("decimal(5,2)"),
                F.col("Sudden_Braking_Count").cast("int"),
                F.col("Rapid_Acceleration_Count").cast("int"),
                F.col("Harsh_Cornering_Count").cast("int"),
                F.col("Speeding_Incidents").cast("int"),
                F.col("Safety_Score").cast("decimal(5,2)"),
                F.col("Braking_Index").cast("decimal(5,2)"),
                F.col("Acceleration_Index").cast("decimal(5,2)"),
                F.col("Trip_Risk_Level").cast("string")
            )
            
            # Append to existing table - will raise exception on error
            spark_batch_df.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable("driver_telemetry_data")
        
        # Move to next batch of policies
        policy_index += batch_size
        
        # Get current table count
        table_count = spark.sql("SELECT COUNT(*) as count FROM driver_telemetry_data").collect()[0]['count']

        
        # Display current batch information
        clear_output(wait=True)
        print(f"=== Fabric Telemetry Generation & Eventhouse Ingestion (REST API) ===")
        print(f"Iteration: {iteration}")
        print(f"Policies Processed: {policy_index}/{len(policies_to_process)}")
        print(f"Progress: {(policy_index/len(policies_to_process)*100):.1f}%")
        if len(new_trips) > 0:
            # Parse datetime for display
            first_trip_time = pd.to_datetime(new_trips[0]['Start_Time'])
            last_trip_time = pd.to_datetime(new_trips[-1]['Start_Time'])
            print(f"Batch Date Range: {first_trip_time.strftime('%Y-%m-%d')} to {last_trip_time.strftime('%Y-%m-%d')}")
        print(f"Trips Generated in Batch: {len(new_trips)}")
        print(f"✓ Sent to Eventhouse (REST API): {len(new_trips)}")
        if skipped_in_batch > 0:
            print(f"⚠ Trips Skipped (time overlap): {skipped_in_batch}")
        print(f"Total Trips Skipped (session): {total_skipped}")
        print(f"Total Trips Sent to Eventhouse: {total_sent_to_eventhouse}")
        print(f"Total Trips in Delta Table: {table_count}")
        print(f"Total Trips Generated This Session: {len(telemetry_records)}")
        
        if len(new_trips) > 0:
            print(f"\nLast 5 Trips Generated:")
            # Show summary of last batch
            for trip in new_trips[-5:]:
                trip_time = pd.to_datetime(trip['Start_Time'])
                print(f"  - PH {trip['policyholder_id']}, VIN: {trip['vin'][-6:]}, "
                      f"Date: {trip_time.strftime('%Y-%m-%d %H:%M')}, "
                      f"Safety: {trip['Safety_Score']}, Risk: {trip['Trip_Risk_Level']}")
        
        if policy_index < len(policies_to_process):
            print(f"\nWaiting 5 seconds before processing next batch...")
            print(f"Press 'Interrupt Kernel' to stop")
            time.sleep(5)
        else:
            print("\n✓ All policies processed!")
            print(f"✓ Total trips sent to Eventhouse via REST API: {total_sent_to_eventhouse}")
        
except KeyboardInterrupt:
    print("\n\n=== Generation Stopped by User ===")
    print(f"Policies processed: {policy_index}/{len(policies_to_process)}")
    print(f"Total trips generated this session: {len(telemetry_records)}")
    print(f"Total trips sent to Eventhouse: {total_sent_to_eventhouse}")
    print(f"Total trips skipped (overlaps): {total_skipped}")
    
    # Get final table count
    try:
        final_count = spark.sql("SELECT COUNT(*) as count FROM driver_telemetry_data").collect()[0]['count']
        print(f"Total records in Driver_Telemetry_Data table: {final_count}")
    except:
        pass

=== Fabric Telemetry Generation & Eventhouse Ingestion (REST API) ===
Iteration: 10
Policies Processed: 82/82
Progress: 100.0%
Batch Date Range: 2018-08-23 to 2019-05-22
Trips Generated in Batch: 17
✓ Sent to Eventhouse (REST API): 17
Total Trips Skipped (session): 0
Total Trips Sent to Eventhouse: 295
Total Trips in Delta Table: 869
Total Trips Generated This Session: 295

Last 5 Trips Generated:
  - PH 54, VIN: 109292, Date: 2020-04-16 10:18, Safety: 97.92, Risk: Low
  - PH 54, VIN: 109293, Date: 2019-08-12 00:29, Safety: 97.45, Risk: Low
  - PH 54, VIN: 109293, Date: 2019-11-07 02:40, Safety: 83.86, Risk: Low
  - PH 54, VIN: 109293, Date: 2019-01-27 14:28, Safety: 92.43, Risk: Low
  - PH 54, VIN: 109293, Date: 2019-05-22 17:53, Safety: 99.59, Risk: Low

✓ All policies processed!
✓ Total trips sent to Eventhouse via REST API: 295


In [14]:
# Optional: View the accumulated telemetry data
if 'telemetry_records' in globals() and len(telemetry_records) > 0:
    telemetry_df = pd.DataFrame(telemetry_records)
    print(f"Total trips in memory: {len(telemetry_df)}")
    print("\n=== Summary Statistics ===")
    print(f"Date Range: {telemetry_df['Start_Time'].min()} to {telemetry_df['Start_Time'].max()}")
    print(f"\nRisk Level Distribution:")
    print(telemetry_df['Trip_Risk_Level'].value_counts())
    print(f"\nTime of Day Distribution:")
    print(telemetry_df['Time_of_Day_Category'].value_counts())
    print(f"\nAverage Safety Score: {telemetry_df['Safety_Score'].mean():.2f}")
    
    # Display last 10 trips
    print("\n=== Last 10 Trips ===")
    display(telemetry_df.tail(10))
else:
    print("No telemetry data generated yet. Run the cell above to start generation.")

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 16, Finished, Available, Finished)

Total trips in memory: 298

=== Summary Statistics ===
Date Range: 2015-03-16T09:43:00.000Z to 2025-12-16T04:09:00.000Z

Risk Level Distribution:
Trip_Risk_Level
High      119
Low       100
Medium     79
Name: count, dtype: int64

Time of Day Distribution:
Time_of_Day_Category
Night        108
Morning       88
Afternoon     52
Evening       50
Name: count, dtype: int64

Average Safety Score: 61.57

=== Last 10 Trips ===


SynapseWidget(Synapse.DataFrame, c742429b-4849-4d0e-ad43-f2e99b1aa0eb)

In [ ]:
# Optional: Export policy-aligned telemetry data to CSV
if 'telemetry_records' in globals() and len(telemetry_records) > 0:
    telemetry_df = pd.DataFrame(telemetry_records)
    
    # Generate filename with date range
    if isinstance(telemetry_df['Start_Time'].iloc[0], datetime):
        min_date = telemetry_df['Start_Time'].min().strftime('%Y%m%d')
        max_date = telemetry_df['Start_Time'].max().strftime('%Y%m%d')
    else:
        min_date = pd.to_datetime(telemetry_df['Start_Time']).min().strftime('%Y%m%d')
        max_date = pd.to_datetime(telemetry_df['Start_Time']).max().strftime('%Y%m%d')
    
    output_file = f'c:\\synthetic_data\\driver_telemetry_policy_aligned_{min_date}_to_{max_date}.csv'
    
    # Convert datetime objects to strings for CSV export
    export_df = telemetry_df.copy()
    if isinstance(export_df['Start_Time'].iloc[0], datetime):
        export_df['Start_Time'] = export_df['Start_Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
        export_df['End_Time'] = export_df['End_Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
    
    export_df.to_csv(output_file, index=False)
    print(f"Policy-aligned telemetry data exported to: {output_file}")
    print(f"Total records exported: {len(export_df)}")
    print(f"Date range: {min_date} to {max_date}")
else:
    print("No telemetry data available to export.")

# Verify the Delta table exists
try:
    row_count = spark.sql("SELECT COUNT(*) as count FROM Driver_Telemetry_Data").collect()[0]['count']
    print(f"✓ Driver_Telemetry_Data table exists with {row_count} existing records")
except Exception as e:
    print(f"⚠ Table does not exist or error occurred: {e}")
    print("Creating table...")
    spark.sql("""
    CREATE TABLE IF NOT EXISTS Driver_Telemetry_Data (
        -- Primary Key and Identification
        Trip_ID STRING NOT NULL,
        policyholder_id INT NOT NULL,
        vin STRING NOT NULL,

        -- Time and Location Context
        Start_Time TIMESTAMP NOT NULL,
        End_Time TIMESTAMP,
        Start_Location_Lat DECIMAL(9, 6),
        Start_Location_Lon DECIMAL(9, 6),
        Time_of_Day_Category STRING,

        -- Basic Trip Measurements
        Duration_Min DECIMAL(10, 2),
        Distance_Miles DECIMAL(10, 2),

        -- Speed Metrics
        Peak_High_Speed DECIMAL(5, 2),
        Average_Speed DECIMAL(5, 2),
        Peak_Low_Speed DECIMAL(5, 2),
        Max_Speed_Limit_Exceeded DECIMAL(5, 2),

        -- Event Counts (Habits)
        Sudden_Braking_Count INT,
        Rapid_Acceleration_Count INT,
        Harsh_Cornering_Count INT,
        Speeding_Incidents INT,

        -- Calculated/Derived Metrics
        Safety_Score DECIMAL(5, 2),
        Braking_Index DECIMAL(5, 2),
        Acceleration_Index DECIMAL(5, 2),
        Trip_Risk_Level STRING
    )
    USING DELTA
    """)
    print("✓ Table created successfully")


# Optional: Insert accumulated in-memory data to table (if generation was stopped)
if 'telemetry_records' in globals() and len(telemetry_records) > 0:
    print(f"Found {len(telemetry_records)} records in memory from this session")
    
    # Create DataFrame from in-memory records
    telemetry_df = pd.DataFrame(telemetry_records)
    
    # Convert pandas datetime strings to proper datetime if needed
    if telemetry_df['Start_Time'].dtype == 'object':
        telemetry_df['Start_Time'] = pd.to_datetime(telemetry_df['Start_Time'])
        telemetry_df['End_Time'] = pd.to_datetime(telemetry_df['End_Time'])
    
    # Convert to Spark DataFrame with proper types
    spark_telemetry_df = spark.createDataFrame(telemetry_df)
    spark_telemetry_df = spark_telemetry_df.select(
        F.col("Trip_ID").cast("string"),
        F.col("policyholder_id").cast("int"),
        F.col("vin").cast("string"),
        F.col("Start_Time").cast("timestamp"),
        F.col("End_Time").cast("timestamp"),
        F.col("Start_Location_Lat").cast("decimal(9,6)"),
        F.col("Start_Location_Lon").cast("decimal(9,6)"),
        F.col("Time_of_Day_Category").cast("string"),
        F.col("Duration_Min").cast("decimal(10,2)"),
        F.col("Distance_Miles").cast("decimal(10,2)"),
        F.col("Peak_High_Speed").cast("decimal(5,2)"),
        F.col("Average_Speed").cast("decimal(5,2)"),
        F.col("Peak_Low_Speed").cast("decimal(5,2)"),
        F.col("Max_Speed_Limit_Exceeded").cast("decimal(5,2)"),
        F.col("Sudden_Braking_Count").cast("int"),
        F.col("Rapid_Acceleration_Count").cast("int"),
        F.col("Harsh_Cornering_Count").cast("int"),
        F.col("Speeding_Incidents").cast("int"),
        F.col("Safety_Score").cast("decimal(5,2)"),
        F.col("Braking_Index").cast("decimal(5,2)"),
        F.col("Acceleration_Index").cast("decimal(5,2)"),
        F.col("Trip_Risk_Level").cast("string")
    )
    
    # Append to existing table (not overwrite)
    print("Appending data to Driver_Telemetry_Data table...")
    spark_telemetry_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("driver_telemetry_data")
    
    print("✓ Data appended successfully!")
    
    # Verify count
    final_count = spark.sql("SELECT COUNT(*) as count FROM driver_telemetry_data").collect()[0]['count']
    print(f"Total records in table: {final_count}")
else:
    print("No telemetry records found in memory. The real-time generator already inserted data directly to the table.")


In [15]:
# Check current table statistics
row_count = spark.sql("SELECT COUNT(*) as count FROM driver_telemetry_data").collect()[0]['count']
print(f"✓ Total records in Driver_Telemetry_Data: {row_count}")

# Show latest records
print("\n=== Latest 10 Records in Table ===")
latest_records = spark.sql("""
    SELECT Trip_ID, policyholder_id, vin, Start_Time, Time_of_Day_Category, 
           Safety_Score, Trip_Risk_Level
    FROM driver_telemetry_data
    ORDER BY Start_Time DESC
    LIMIT 10
""")
latest_records.show(truncate=False)

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 17, Finished, Available, Finished)

✓ Total records in Driver_Telemetry_Data: 574

=== Latest 10 Records in Table ===
+--------------------------------+---------------+-----------------+-------------------+--------------------+------------+---------------+
|Trip_ID                         |policyholder_id|vin              |Start_Time         |Time_of_Day_Category|Safety_Score|Trip_Risk_Level|
+--------------------------------+---------------+-----------------+-------------------+--------------------+------------+---------------+
|1_109187_POL4010_1766551560_0015|1              |2HGBH41JXMN109187|2025-12-24 04:46:00|Night               |36.38       |High           |
|3_109191_POL4030_1765858140_0021|3              |6HGBH41JXMN109191|2025-12-16 04:09:00|Night               |96.01       |Low            |
|1_109187_POL4010_1765255800_0019|1              |2HGBH41JXMN109187|2025-12-09 04:50:00|Night               |38.09       |High           |
|3_109191_POL4030_1764915600_0018|3              |6HGBH41JXMN109191|2025-12-05 06:20

In [37]:
# Analyze driver profiles
profile_summary = spark.sql("""
SELECT 
    policyholder_id,
    COUNT(*) as trip_count,
    ROUND(AVG(Safety_Score), 2) as avg_safety_score,
    ROUND(AVG(Average_Speed), 2) as avg_speed,
    FIRST(Trip_Risk_Level) as risk_level,
    ROUND(AVG(Speeding_Incidents), 2) as avg_speeding_incidents,
    ROUND(AVG(Sudden_Braking_Count), 2) as avg_sudden_braking
FROM Driver_Telemetry_Data
GROUP BY policyholder_id
ORDER BY policyholder_id
LIMIT 20
""")

profile_summary.show()


StatementMeta(, b5ea4c1a-85c3-472e-aeeb-4b6bb891b829, 42, Finished, Available, Finished)

+---------------+----------+----------------+---------+----------+----------------------+------------------+
|policyholder_id|trip_count|avg_safety_score|avg_speed|risk_level|avg_speeding_incidents|avg_sudden_braking|
+---------------+----------+----------------+---------+----------+----------------------+------------------+
|              1|        84|           35.53|    65.21|      High|                  9.48|             14.74|
|             10|        90|           35.30|    64.89|      High|                  9.51|             14.09|
|              2|        54|           64.26|    52.05|    Medium|                  3.87|              5.17|
|              3|        74|           91.23|    39.88|       Low|                  0.85|              1.31|
|              4|        67|           34.71|    65.76|      High|                 10.19|             14.04|
|              5|        79|           65.24|    52.38|    Medium|                  4.09|              5.57|
|             51|  

In [16]:
# Display profile type distribution
print("\n=== Driver Behavior Distribution ===")
for pid in sorted(telemetry_df['policyholder_id'].unique())[:15]:
    profile = assign_driver_profile(int(pid))
    count = len(telemetry_df[telemetry_df['policyholder_id'] == pid])
    avg_safety = telemetry_df[telemetry_df['policyholder_id'] == pid]['Safety_Score'].mean()
    avg_speed = telemetry_df[telemetry_df['policyholder_id'] == pid]['Average_Speed'].mean()
    print(f"Policyholder {pid}: {profile.upper()} driver - {count} trips, Safety: {avg_safety:.1f}, Speed: {avg_speed:.1f} mph")

StatementMeta(, de1ab0d6-0b8d-4039-9dfa-ef004232aaed, 18, Finished, Available, Finished)


=== Driver Behavior Distribution ===
Policyholder 1: AGGRESSIVE driver - 49 trips, Safety: 34.8, Speed: 66.0 mph
Policyholder 2: MODERATE driver - 47 trips, Safety: 67.5, Speed: 52.6 mph
Policyholder 3: SAFE driver - 49 trips, Safety: 88.5, Speed: 40.2 mph
Policyholder 4: AGGRESSIVE driver - 41 trips, Safety: 35.5, Speed: 65.4 mph
Policyholder 5: MODERATE driver - 9 trips, Safety: 66.0, Speed: 51.2 mph
Policyholder 6: SAFE driver - 12 trips, Safety: 90.9, Speed: 43.7 mph
Policyholder 7: AGGRESSIVE driver - 12 trips, Safety: 32.9, Speed: 65.2 mph
Policyholder 8: MODERATE driver - 12 trips, Safety: 66.7, Speed: 53.5 mph
Policyholder 9: SAFE driver - 9 trips, Safety: 88.8, Speed: 37.7 mph
Policyholder 10: AGGRESSIVE driver - 9 trips, Safety: 35.5, Speed: 62.0 mph
Policyholder 51: SAFE driver - 18 trips, Safety: 89.2, Speed: 41.3 mph
Policyholder 52: AGGRESSIVE driver - 8 trips, Safety: 32.9, Speed: 68.1 mph
Policyholder 53: MODERATE driver - 11 trips, Safety: 64.6, Speed: 54.4 mph
Policy

# Load the driver_telemetry_data table as a Spark DataFrame
spark_df = spark.read.table("driver_telemetry_data")

# Export the DataFrame to a CSV file in the Files/ directory, with column names as headers
# The output folder will be: /lakehouse/default/Files/driver_telemetry_data_csv_export
spark_df.write.csv('Files/AutoClaims_csv/output/driver_telemetry_data_csv', header=True, mode="overwrite")

# By default, write.csv() generates multiple part-files (one per partition). To get a single CSV file, coalesce to 1 partition BEFORE write:
single_file_df = spark.read.table("driver_telemetry_data").coalesce(1)

# Write the DataFrame as a single file (with headers)
# The file will appear inside the directory 'Files/AutoClaims_csv/output/driver_telemetry_data_csv_single'
single_file_df.write.csv('Files/AutoClaims_csv/output/driver_telemetry_data_csv', header=True, mode="overwrite")

# Note:
# After running, you will find a directory 'driver_telemetry_data_csv_single' with a single CSV file (part-00000...) and some metadata files.
# To get a file with a custom name (not 'part-00000...'), you must use additional steps: move/rename the file using filesystem utilities.